In [33]:
!pip install gradio

In [34]:
!pip install flair

In [35]:
import gradio as gr
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

from flair.models import SequenceTagger
from flair.data import Sentence

import threading
from concurrent.futures import thread
import os
import random

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [36]:
flair_model = SequenceTagger.load('kaliani/flair-ner-skill')

2025-11-21 10:40:29,129 SequenceTagger predicts: Dictionary with 7 tags: O, S-SKILL, B-SKILL, E-SKILL, I-SKILL, <START>, <STOP>


In [37]:
experience_level_mapping = {
    "Internship" : "f_E=1",
    "Entry level" : "f_E=2",
    "Associate" : "f_E=2",
    "Mid-Senior Level" : "f_E=2",
}

work_type_mapping = {
    "On-site" : "f_WT=1",
    "Hybrid" : "f_WT=2",
    "Remote" : "f_WT=3"
}

time_filter_mapping = {
    "Past 24 hours" : "f_TPR=r86400",
    "Past week" : "f_TPR=r604800",
    "Past month" : "f_TPR=r2592000"
}

In [38]:
description = """We are looking for a creative and detail-oriented Game Designer to contribute to the development of compelling gameplay systems, engaging levels, and memorable player experiences. You will work closely with designers, artists, and programmers to define and refine core game mechanics, level layouts, and overall gameplay flow.



This role is ideal for someone who is passionate about crafting unique gameplay ideas, solving design challenges.



Key Responsibilities:

Design and iterate on core gameplay mechanics, combat systems, puzzles, and player interactions.
Create level layouts, flowcharts, blockouts, and gameplay scenarios that support the game’s vision and narrative.
Work closely with artists and programmers to bring design concepts to life while maintaining clarity and consistency.
Develop and maintain design documentation, including feature specs, level briefs, system overviews, and gameplay balancing notes.
Playtest features and levels regularly, identifying issues and proposing creative solutions.
Ensure that gameplay systems are intuitive, engaging, and aligned with the player experience goals.
Collaborate with cross-functional teams to meet production milestones within an agile workflow.


Technical Skills:

Strong understanding of game design principles, player psychology, difficulty balancing, and pacing.
Experience working with Unreal Engine for level design or gameplay prototyping.
Ability to create clear documentation, level maps, design specs, and mockups.
Familiarity with scripting or blueprint logic (for prototyping gameplay and interactions).
Good understanding of camera systems, navigation, combat design, and action-adventure mechanics is a plus.


Soft Skills:

Strong creative thinking and problem-solving abilities.
Excellent communication and collaboration skills across disciplines.
Ability to receive and incorporate feedback constructively.
Highly organized with strong attention to detail.
Passion for game design, world-building, and creating player experiences.


Preferred Additional Skills:

Experience designing levels for action, platforming, puzzle, or narrative-driven games.
Familiarity with blockout workflows and greyboxing in Unreal.
Understanding of combat systems, enemy behavior design, and encounter pacing.
Ability to prototype gameplay mechanics using Blueprints.
Experience with planning tools such as Miro, Figma, or other design visualization tools.


Education & Experience:

Bachelor’s degree in Game Design, Computer Science, Interactive Media, or a related field (or equivalent practical experience).
Professional or project-based experience in game design, level design, or gameplay systems design.


How to Apply:

Please submit your CV/Resume along with a portfolio (PDF format or direct links—shortened URLs are not accepted) to martin@rickshawgamestudios.com. A cover letter is optional but recommended."""

In [39]:
def get_skills(text):
    sentence = Sentence(text)
    flair_model.predict(sentence)
    return [entity.text for entity in sentence.get_spans('ner')]

get_skills(description)

['Game Design', 'Computer Science', 'Interactive Media', '.']

In [40]:
class ScrapperManager:
  def __init__(self):

    #setup when class starts
    self.stop_event = threading.Event()
    #empty table for job data
    self.current_df = pd.DataFrame()
    #lock
    self.lock = threading.Lock()

  def reset(self):
    self.stop_event.clear()
    self.current_df = pd.DataFrame()

  def add_job(self, job_data):
    with self.lock:
      new_df = pd.DataFrame([job_data])
      self.current_df = pd.concat([self.current_df, new_df], ignore_index=True)

scraper_manager = ScrapperManager()

### Save to CSV Function

In [41]:
def save_csv(df, filename="jobs"):
  try:
    os.makedirs("saved_jobs", exist_ok=True)

    if not filename:
      filename = f"jobs_{int(time.time())}"
    full_path = f"saved_jobs/{filename}.csv"
    df.to_csv(full_path, index=False)

    return f"Saved to {full_path}"
  except Exception as e:
    return f"Error : {str(e)}"


#### Process Job Function

In [42]:
def process_job(job, work_type, exp_level, position):
    try:
        title_element = job.find('h3', class_='base-search-card__title')
        company_element = job.find('a', class_='hidden-nested-link')
        loc_element = job.find('span', class_='job-search-card__location')
        link_element = job.find('a', class_='base-card__full-link')

        if not all([title_element, company_element, loc_element, link_element]):
            return None

        title = title_element.text.strip()
        company = company_element.text.strip()
        loc = loc_element.text.strip()
        link = link_element['href'].split('?')[0]

        session = requests.Session()
        retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        session.mount('https://', HTTPAdapter(max_retries=retries))

        desc = "Description not available"
        skills = []

        try:
            time.sleep(random.uniform(2, 5))

            response = session.get(
                link,
                headers={
                    'User-Agent': random.choice([
                        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15',
                        'Mozilla/5.0 (Macintosh; X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36'
                    ]),
                    'Accept-Language': 'en-US,en;q=0.9'
                },
                timeout=10
            )

            job_soup = BeautifulSoup(response.text, 'html.parser')

            description_selectors = [
                'div.description__text',
                'div.show-more-less-html__markup',
                'div.core-section-container__content',
                'section.core-section-container'
            ]

            for selector in description_selectors:
                desc_element = job_soup.select_one(selector)
                if desc_element:
                    desc = desc_element.get_text('\n').strip()
                    skills = get_skills(desc)
                    break
        except Exception as e:
            print(f"Error fetching job description: {str(e)}")

        return {
            "Position": position,
            "Date": datetime.now().strftime("%Y-%m-%d"),
            "Work type": work_type,
            "Level": exp_level,
            "Title": title,
            "Company": company,
            "Location": loc,
            "Link": f"[{link}{link})",
            "Description": desc,
            "Skills": ", ".join(skills[:5]) if skills else "No skills detected"
        }

    except Exception as e:
        print(f"Error processing job: {str(e)}")
        return None

#### Scrape Jobs

In [43]:
def scrape_jobs(location, position, work_types, exp_levels, time_filter):

  session = requests.Session()
  retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])

  session.mount('https://', HTTPAdapter(max_retries=retries))

  for work_type in work_types:
    for exp_level in exp_levels:
      if scraper_manager.stop_event.is_set():
        return

      try:
        base_url = f"https://www.linkedin.com/jobs/search/?keywords={position}&location={location}"\
                    f"&{work_type_mapping[work_type]}" \
                    f"&{experience_level_mapping[exp_level]}" \
                    f"&{time_filter_mapping[time_filter]}" \
                    f"&radius=0"
        try:
          #get search page
          response = session.get(base_url, timeout=10)

          #parse HTML
          soup = BeautifulSoup(response.text, 'html_parser')

          total_jobs = int(soup.find('span', class_='results-context-header__job-count')).text.replace(',', '')

        except:
          total_jobs=25

        total_jobs = min(total_jobs, 100)

        for start in range(0, total_jobs, 25):
          if scraper_manager.stop_event.is_set():
            return

          time.sleep(random.uniform(2,5))

          url = f"{base_url}&start={start}"

          try:
            response = session.get(url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            jobs = soup.find_all('div', class_='base-card')
          except Exception as e:
              print(f"Failed to scrape page {start} : {str(e)}")
              continue

          random.shuffle(jobs)

          for job in jobs:
            if scraper_manager.stop_event.is_set():
              return

            job_data = process_job(job, work_type, exp_level, position)
            if job_data:
              scraper_manager.add_job(job_data)
              yield


      except Exception as e:
        print(f"Scrapping error: {str(e)}")



In [44]:
def run_scrapper(cities, states, positions, work_types, exp_level, time_filter):
  scraper_manager.reset()

  cities_list = [c.strip() for c in cities.split(',') if c.strip()]
  states_list = [s.strip() for s in states.split(',') if s.strip()]
  locations = [f"{city}, {state}" for city in cities_list for state in states_list]
  positions = [p.strip().replace(' ', '%20') for p in positions.aplit(',') if p.strip()]


  def worker():
    for loc in locations:
      for pos in positions:
        if scraper_manager.stop_event.is_set():
          return


  thread = threading.Thread(target=worker)
  thread.start()

  while thread.is_alive():
    time.sleep(0.5)

  with scraper_manager.lock:
    yield 'Scrapping in progress...', scraper_manager.current_df

  yield "Scraping Completed" if not scraper_manager.stop_event.is_set() else "Scraping Stopped! ", scraper_manager.current_df



In [45]:
def run_scrapper(cities, states, positions, work_types, exp_level, time_filter):
    scraper_manager.reset()

    # 1. Cleaning inputs
    cities_list = [c.strip() for c in cities.split(',') if c.strip()]
    states_list = [s.strip() for s in states.split(',') if s.strip()]
    locations = [f"{city}, {state}" for city in cities_list for state in states_list]
    # Fix: typo 'aplit' to 'split'
    positions_list = [p.strip().replace(' ', '%20') for p in positions.split(',') if p.strip()]

    # 2. The Worker Logic
    def worker():
        for loc in locations:
            for pos in positions_list:
                if scraper_manager.stop_event.is_set():
                    return
                # ACTUAL FIX: We must iterate through the generator here
                # Note: We convert the generator to a list or loop it to trigger execution
                generator = scrape_jobs(loc, pos, work_types, exp_level, time_filter)
                for _ in generator:
                    if scraper_manager.stop_event.is_set():
                        return

    # 3. Threading
    t = threading.Thread(target=worker)
    t.start()

    # 4. Yielding updates to Gradio
    while t.is_alive():
        time.sleep(1) # Update every second
        with scraper_manager.lock:
            # Yield status and the current dataframe
            yield f"Scraping {len(scraper_manager.current_df)} jobs...", scraper_manager.current_df

    yield "Scraping Completed", scraper_manager.current_df

#### Gradio Interface

In [46]:
with gr.Blocks() as app:
  gr.Markdown("""
    <div
        style = 'text-align: center;
                  color:#f67d3c;
                  font-size: 2em;
                  font-weight: bold;
                  margin: 20px 0;
                  padding: 10px'
    >

    LinkedIn Job Scrapper
    </div>
  """)

  with gr.Row():
    with gr.Column():
      cities = gr.Textbox(label="Cities (comma-separated)")
      states = gr.Textbox(label="States/Countries (comma-separated)")
      positions = gr.Textbox(label="Positions (comma-separated)")
      work_types = gr.CheckboxGroup(list(work_type_mapping.keys()), label="Work Types")
      exp_levels = gr.CheckboxGroup(list(experience_level_mapping.keys()), label="Experience Levels")
      time_filter = gr.Dropdown(list(time_filter_mapping.keys()), label="Time Filter")

      with gr.Row():
        start_btn = gr.Button("Start Scraping", variant="primary")
        stop_btn = gr.Button("Stop Scraping", variant="secondary")

  status = gr.Textbox(label="Status")
  results = gr.DataFrame(
      headers = ["Position", "Date", "Work type", "Level", "Title", "Company", "Location", "Link", "Skills"],
      datatype = [  "str", "str", "str", "str", "str", "str", "str", "markdown", "str"],
      interactive = False
  )

  with gr.Row():
    filename = gr.Textbox(label="Filename (optional)", placeholder="my_jobs")
    save_btn = gr.Button("Save to CSV", variant="secondary")

  save_status = gr.Textbox(label="Save Status")


  start_btn.click(
      run_scrapper,
      inputs=[cities, states, positions, work_types, exp_levels, time_filter],
      outputs=[status, results]
      )

  stop_btn.click(
      lambda : scraper_manager.stop_event.set(),
      outputs = None
  )

  save_btn.click(
      save_csv,
      inputs = [results, filename],
      outputs = save_status
      )

if __name__ == "__main__":
  app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc348411d895b27d42.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
